# EA1 – Carga y creación de dataset

Kevin Daniel Sepúlveda Giraldo · Analítica de Datos · PREICA2602B020125

Dataset: [`aptologia/casa-vs-apartamento-precio-m2-ciudades-colombia-2026`](https://huggingface.co/datasets/aptologia/casa-vs-apartamento-precio-m2-ciudades-colombia-2026): precio por m² de casa y de apartamento en ciudades de Colombia (2026).

## 1. Importar las clases

In [ ]:
import pandas as pd

from cargador_hf import CargadorHuggingFace
from resumen_dataframe import ResumenDataFrame

## 2. Cargar el dataset

`max_filas` limita cuántas filas se descargan.

In [ ]:
DATASET = "aptologia/casa-vs-apartamento-precio-m2-ciudades-colombia-2026"

cargador = CargadorHuggingFace(DATASET, split="train", max_filas=1000)
df_original = cargador.cargar()

print(cargador.informacion())
df_original.head()

In [ ]:
df_original.info()

## 3. Transformaciones

### 3.1 Renombrar columnas

Nombres en español y sin abreviaturas.

In [ ]:
df_t1 = df_original.rename(columns={
    "city": "ciudad",
    "casa_cop_m2": "precio_casa_m2",
    "apto_cop_m2": "precio_apto_m2",
    "diferencia_pct": "diferencia_casa_vs_apto_pct",
})
df_t1.columns.tolist()

### 3.2 Seleccionar columnas

Quito `n_si` y `n_no`: la ficha del dataset no explica qué son.

In [ ]:
df_t2 = df_t1[["ciudad", "precio_casa_m2", "precio_apto_m2", "diferencia_casa_vs_apto_pct"]]
print(f"Columnas antes: {df_t1.shape[1]} | Columnas después: {df_t2.shape[1]}")
df_t2.head()

### 3.3 Columnas nuevas

Precios en millones de pesos y `tipo_mas_economico` (diferencia negativa = la casa es más barata).

In [ ]:
df_t3 = df_t2.assign(
    precio_casa_m2_millones=(df_t2["precio_casa_m2"] / 1_000_000).round(2),
    precio_apto_m2_millones=(df_t2["precio_apto_m2"] / 1_000_000).round(2),
    diferencia_casa_vs_apto_pct=df_t2["diferencia_casa_vs_apto_pct"].round(2),
    tipo_mas_economico=df_t2["diferencia_casa_vs_apto_pct"].apply(
        lambda dif: "Sin dato" if pd.isna(dif) else ("Casa" if dif < 0 else "Apartamento")
    ),
)
df_t3.head()

### 3.4 Quitar las columnas en pesos y ordenar

Primero las ciudades donde la casa sale más barata frente al apartamento.

In [ ]:
df_transformado = (
    df_t3.drop(columns=["precio_casa_m2", "precio_apto_m2"])
         .sort_values("diferencia_casa_vs_apto_pct")
         .reset_index(drop=True)
)
df_transformado.head(10)

### Antes y después

In [ ]:
print(f"Original:     {df_original.shape[0]} filas x {df_original.shape[1]} columnas")
print(f"Transformado: {df_transformado.shape[0]} filas x {df_transformado.shape[1]} columnas")
print("Columnas finales:", df_transformado.columns.tolist())

## 4. Resumen en `.txt`

In [ ]:
resumen = ResumenDataFrame(
    df_transformado,
    titulo="Resumen: precio m2 casa vs apartamento en ciudades de Colombia (2026)",
)
ruta_resumen = resumen.guardar("resultados/resumen_casa_vs_apartamento.txt")

print(f"Resumen guardado en: {ruta_resumen}\n")
print(ruta_resumen.read_text(encoding="utf-8"))

## 5. Resultado

In [ ]:
conteo = df_transformado["tipo_mas_economico"].value_counts()
print("Ciudades donde el m2 es más barato según el tipo de vivienda:")
print(conteo.to_string())

mayor = df_transformado.iloc[0]
print(f"\nMayor diferencia a favor de la casa: {mayor['ciudad']} "
      f"({mayor['diferencia_casa_vs_apto_pct']}%)")